In [333]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import LeakyReLU



In [194]:
dataset = pd.read_csv('C:/Users/Pavel/Lessen_jypyter/csv/cardiovascular_risk_dataset.csv')

In [328]:
dataset

,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week,risk_category
0,62,25.0,142,93,247,72,2.0,11565,3,5.6,8.2,0.0,7,0.7,1
1,54,29.7,158,101,254,74,0.0,4036,8,0.5,6.7,0.0,5,4.5,2
2,46,36.2,170,113,276,80,0.0,3043,9,0.4,4.0,0.0,1,20.8,2
3,48,30.4,153,98,230,73,1.0,5604,5,0.6,8.0,0.0,4,8.5,1
4,46,25.3,139,87,206,69,0.0,7464,1,2.0,6.1,0.0,5,3.6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,19,26.0,121,75,185,84,2.0,6724,3,2.9,7.2,0.0,7,0.0,0
5496,18,30.9,128,82,235,75,2.0,3661,4,0.0,5.5,0.0,1,9.6,0
5497,63,29.5,142,92,239,69,2.0,6643,5,4.1,6.9,0.0,6,2.4,1
5498,46,27.5,138,91,237,65,2.0,3279,3,2.4,5.8,1.0,5,2.3,1


In [196]:
dataset = dataset.drop(['Patient_ID', 'heart_disease_risk_score'], axis = 1)

In [197]:
dataset['smoking_status'] = dataset['smoking_status'].map({'Never': 2,'Former': 1, 'Current': 0}).astype(float)
dataset['family_history_heart_disease'] = dataset['family_history_heart_disease'].map({'Yes': 1, 'No': 0}).astype(float)
dataset['risk_category'] = dataset['risk_category'].map({'High': 2,'Medium': 1, 'Low': 0})

In [198]:
target = dataset['risk_category']
feature = dataset.drop(['risk_category'], axis = 1)

In [199]:
X_train, X_test, y_train, y_test = train_test_split(feature, target , random_state = 42, test_size = .2)

In [226]:
class split_data():
    def __init__(self,feature_train, feature_test, target_train, target_test, batch = 64):
        self.feature_train = torch.FloatTensor(feature_train.values)
        self.target_train = torch.LongTensor(target_train.values)
        self.feature_test = torch.FloatTensor(feature_test.values)
        self.target_test = torch.LongTensor(target_test.values)
        self.batch = batch
        
    def train_test(self):
        train = DataLoader(
            TensorDataset(self.feature_train, self.target_train),
            batch_size = self.batch,
            shuffle = True
        )
        test = DataLoader(
            TensorDataset(self.feature_test, self.target_test),
            batch_size = self.batch,
            shuffle = False
        )
        incol = self.feature_train.shape[1]
        return train, test, incol

tts = split_data(X_train,X_test,y_train,y_test)
trainloader , testloader , incol= tts.train_test()


In [249]:
class NeuralModelTorch(nn.Module):
    def __init__(self,
                input_size = 14, 
                hidden_size1 = 128,
                hidden_size2 = 64,
                hidden_size3 = 32,
                output_size = 3
                ):
        super(NeuralModelTorch, self).__init__()
        self.sloi1 = nn.Linear(input_size, hidden_size1)
        self.sloi2 = nn.Linear(hidden_size1, hidden_size2)
        self.activator = nn.ReLU()
        self.drop = nn.Dropout(0.2)
        self.norm = nn.BatchNorm1d(hidden_size2)
        self.sloi3 = nn.Linear(hidden_size2, hidden_size3)
        self.activator2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        self.norm2 = nn.BatchNorm1d(hidden_size3)
        self.sloi4 = nn.Linear(hidden_size3, output_size)
        

    def forward(self, x):
        out = self.sloi1(x)
        out = self.sloi2(out)
        out = self.activator(out)
        out = self.drop(out)
        out = self.norm(out)
        out = self.sloi3(out)
        out = self.activator2(out)
        out = self.drop2(out)
        out = self.norm2(out)
        out = self.sloi4(out)
        return out

    

In [335]:
def split(dataset):
    target = dataset['risk_category']
    features = dataset.drop(['risk_category'], axis = 1)
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = .2, random_state = 42)
    
    incol = X_train.shape[1]
    return X_train, X_test, y_train, y_test, incol

x_train, x_test, y_train, y_test, incol = split(dataset)

In [300]:
def train(loader, model,optimizer,loss_fn):
    model.train()
    size = len(loader.dataset)
    num = len(loader)
    epochs = 20
    train_loss = 0.0
    correct = 0.0 
    total = 0
    
    for epoch in range(epochs):

        for batch, (X , y)  in enumerate(loader):
    
            pred = model(X)
    
            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            total += y.size(0)
    avg_loss = train_loss / num
    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy : {accuracy:.4f}%')


def test(loader, model, loss_fn):
    model.eval()
    size = len(loader.dataset)
    num_batches = len(loader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    accuracy = 100 * correct / size
    print(f"Test Error: \n Accuracy: {accuracy:>0.1f}%, Avg loss: {test_loss:>8f} \n")





In [302]:
train(trainloader, model, optimizer, loss_fn)
test(testloader, model, loss_fn)

Epoch [20/20], Loss: 10.4918, Accuracy : 76.9670%
Test Error: 
 Accuracy: 68.7%, Avg loss: 0.663637 



In [330]:
def model_keras(feature_train, feature_test, target_train, target_test, incol):
    modelKeras = keras.Sequential([
        keras.layers.Dense(128, input_shape = (feature_train.shape[1],)),
        keras.layers.Dense(64, activation = LeakyReLU(alpha = 0.01)),
            keras.layers.Dropout(0.2),
            keras.layers.BatchNormalization(),
        keras.layers.Dense(32, activation = 'relu', kernel_regularizer='l2'),
            keras.layers.Dropout(0.2),
            keras.layers.BatchNormalization(),
        keras.layers.Dense(3, activation = 'softmax')
    ])

    modelKeras.compile(
            optimizer = keras.optimizers.Adam(learning_rate = 0.01),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
    )
    early_stopping = EarlyStopping(
            monitor='val_loss',
            min_delta=0.01,
            patience=10,
            verbose=0,
            baseline=None,
            restore_best_weights=True,
            start_from_epoch=0
    )
    batch = 64
    modelKeras.fit(feature_train, target_train,batch_size = batch, epochs = 200, callbacks = [early_stopping], validation_split = 0.1,verbose = 0)

    train_loss,train_acc = modelKeras.evaluate(feature_train,target_train)
    test_loss,test_acc = modelKeras.evaluate(feature_test,target_test)
    print(f"Результат Тренировочной и тестовой оценки : {train_acc} / {test_acc}")

In [334]:
neural_network = model_keras(x_train, x_test, y_train, y_test, incol)
neural_network

C:\Users\Pavel\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Pavel\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8902 - loss: 0.2972 
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8745 - loss: 0.3007 
Результат Тренировочной и тестовой оценки : 0.8902272582054138 / 0.8745454549789429
